# 코드 분할 (Split code)

`RecursiveCharacterTextSplitter.from_language()` 를 사용하면 다양한 프로그래밍 언어의 코드를 **언어 문법에 맞는 구분자**(클래스, 함수 정의 등)로 분할할 수 있습니다.
`Language` enum을 import하고 언어를 지정하면 됩니다.

> **🔄 최신 버전 기준 변경 사항 (langchain-text-splitters 1.x)**
> - 핵심 API(`Language`, `from_language`, `get_separators_for_language`)는 그대로입니다.
> - **LaTeX 예제 버그 수정**: 원본은 일반 문자열 안에 `\begin`, `\documentclass` 등을 넣었는데, 파이썬에서 `\b` 는 백스페이스 문자로 해석되어
>   `\begin{document}` 가 깨지고, `\d`, `\m`, `\s` 는 Python 3.12+에서 `SyntaxWarning` 을 냅니다. **raw 문자열(`r"""..."""`)** 로 수정했습니다.
> - React(JSX)·Vue·Svelte 코드를 위한 **`JSFrameworkTextSplitter`** 예제를 추가했습니다.
> - Markdown/HTML은 이 방식보다 구조를 이해하는 전용 분할기(06, 07 노트북)가 더 적합하다는 안내를 추가했습니다.

In [ ]:
%pip install -qU langchain-text-splitters

In [ ]:
from langchain_text_splitters import (
    Language,
    RecursiveCharacterTextSplitter,
)

지원되는 언어 전체 목록을 확인합니다.

In [ ]:
[e.value for e in Language]

`get_separators_for_language()` 로 특정 언어에 사용되는 구분자를 확인할 수 있습니다.

In [ ]:
RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON)

## Python

- `language=Language.PYTHON`
- `chunk_size=50`: 각 청크의 최대 크기
- `chunk_overlap=0`: 청크 간 중복 없음

In [ ]:
PYTHON_CODE = """
def hello_world():
    print("Hello, World!")

hello_world()
"""

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=50, chunk_overlap=0
)

In [ ]:
python_docs = python_splitter.create_documents([PYTHON_CODE])
python_docs

In [ ]:
for doc in python_docs:
    print(doc.page_content, end="\n==================\n")

## JS

In [ ]:
JS_CODE = """
function helloWorld() {
  console.log("Hello, World!");
}

helloWorld();
"""

js_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.JS, chunk_size=60, chunk_overlap=0
)

js_docs = js_splitter.create_documents([JS_CODE])
js_docs

## TS

In [ ]:
TS_CODE = """
function helloWorld(): void {
  console.log("Hello, World!");
}

helloWorld();
"""

ts_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.TS, chunk_size=60, chunk_overlap=0
)
ts_docs = ts_splitter.create_documents([TS_CODE])
ts_docs

## 🔄 추가: React(JSX) / Vue / Svelte

`JSFrameworkTextSplitter` 는 코드에 등장하는 **컴포넌트 태그**(`<div`, `<Button` 등)를 자동으로 찾아 구분자로 추가한 뒤,
JS 구분자와 함께 재귀 분할합니다. 프런트엔드 컴포넌트 코드를 컴포넌트 경계에 맞춰 나누고 싶을 때 사용합니다.

In [ ]:
from langchain_text_splitters.jsx import JSFrameworkTextSplitter

JSX_CODE = """
import React from "react";

export function Greeting({ name }) {
  return (
    <div className="greeting">
      <h1>Hello, {name}!</h1>
      <Button onClick={() => alert("clicked")}>Click me</Button>
    </div>
  );
}

export default function App() {
  return <Greeting name="LangChain" />;
}
"""

jsx_splitter = JSFrameworkTextSplitter(chunk_size=120, chunk_overlap=0)
for chunk in jsx_splitter.split_text(JSX_CODE):
    print(chunk, end="\n==================\n")

## Markdown

> 🔄 참고: 이 방식은 Markdown 헤더를 구분자로 쓰긴 하지만 **헤더 정보를 메타데이터로 남기지 않습니다.**
> 문서 구조(헤더 계층)를 보존해야 한다면 `MarkdownHeaderTextSplitter` (06 노트북)를 사용하세요.

In [ ]:
markdown_text = """
# 🦜️🔗 LangChain

⚡ LLM을 활용한 초스피드 애플리케이션 구축 ⚡

## 빠른 설치

```bash
pip install langchain
```

# 빠르게 발전하는 분야의 오픈 소스 프로젝트 입니다. 많관부 🙏
"""

In [ ]:
md_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN,
    chunk_size=60,
    chunk_overlap=0,
)
md_docs = md_splitter.create_documents([markdown_text])
md_docs

## LaTeX

LaTeX는 수식 표현에 널리 쓰이는 문서 작성용 마크업 언어입니다.

🔄 **raw 문자열(`r"""..."""`)** 을 사용합니다. 일반 문자열이면 `\b`(백스페이스) 등 이스케이프 시퀀스로 해석되어 원문이 손상됩니다.

In [ ]:
latex_text = r"""
\documentclass{article}

\begin{document}

\maketitle

\section{Introduction}
% LLM은 방대한 양의 텍스트 데이터로 학습하여 사람과 유사한 언어를 생성할 수 있는 기계 학습 모델의 한 유형입니다.
% 최근 몇 년 동안 LLM은 언어 번역, 텍스트 생성, 감성 분석 등 다양한 자연어 처리 작업에서 상당한 발전을 이루었습니다.

\subsection{History of LLMs}
% 초기 LLM은 1980년대와 1990년대에 개발되었지만, 처리할 수 있는 데이터 양과 당시 사용 가능한 컴퓨팅 능력으로 인해 제한되었습니다.
% 그러나 지난 10년 동안 하드웨어와 소프트웨어의 발전으로 대규모 데이터 세트에 대해 LLM을 학습시킬 수 있게 되었고, 이는 성능의 큰 향상으로 이어졌습니다.

\subsection{Applications of LLMs}
% LLM은 챗봇, 콘텐츠 생성, 가상 어시스턴트 등 산업 분야에서 많은 응용 분야를 가지고 있습니다.
% 또한 언어학, 심리학, 컴퓨터 언어학 연구를 위해 학계에서도 사용될 수 있습니다.

\end{document}
"""

In [ ]:
latex_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.LATEX,  # LaTeX 구분자 사용
    chunk_size=60,
    chunk_overlap=0,
)
latex_docs = latex_splitter.create_documents([latex_text])
latex_docs

## HTML

> 🔄 참고: 태그 단위로 문자열을 자르기만 하므로 헤더 메타데이터가 남지 않습니다.
> 헤더 계층 보존이나 표·목록 보존이 필요하면 `HTMLHeaderTextSplitter` / `HTMLSemanticPreservingSplitter` (07 노트북)를 사용하세요.

In [ ]:
html_text = """
<!DOCTYPE html>
<html>
    <head>
        <title>🦜️🔗 LangChain</title>
        <style>
            body {
                font-family: Arial, sans-serif;
            }
            h1 {
                color: darkblue;
            }
        </style>
    </head>
    <body>
        <div>
            <h1>🦜️🔗 LangChain</h1>
            <p>⚡ Building applications with LLMs through composability ⚡</p>
        </div>
        <div>
            As an open-source project in a rapidly developing field, we are extremely open to contributions.
        </div>
    </body>
</html>
"""

In [ ]:
html_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.HTML,
    chunk_size=60,
    chunk_overlap=0,
)
html_docs = html_splitter.create_documents([html_text])
html_docs

## Solidity

In [ ]:
SOL_CODE = """
pragma solidity ^0.8.20;
contract HelloWorld {
   function add(uint a, uint b) pure public returns(uint) {
       return a + b;
   }
}
"""

sol_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.SOL, chunk_size=128, chunk_overlap=0
)

sol_docs = sol_splitter.create_documents([SOL_CODE])
sol_docs

## C#

In [ ]:
CSHARP_CODE = """
using System;
class Program
{
    static void Main()
    {
        Console.WriteLine("Enter a number (1-5):");
        int input = Convert.ToInt32(Console.ReadLine());
        for (int i = 1; i <= input; i++)
        {
            if (i % 2 == 0)
            {
                Console.WriteLine($"{i} is even.");
            }
            else
            {
                Console.WriteLine($"{i} is odd.");
            }
        }
        Console.WriteLine("Goodbye!");
    }
}
"""

csharp_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.CSHARP, chunk_size=128, chunk_overlap=0
)
csharp_docs = csharp_splitter.create_documents([CSHARP_CODE])
csharp_docs

## 🔄 추가: 실제 소스 파일을 분할할 때

실무에서는 파일 확장자로 언어를 고르고, 파일 경로를 메타데이터로 남겨 두면 검색 결과에서 출처를 추적하기 쉽습니다.

In [ ]:
from pathlib import Path

from langchain_core.documents import Document

EXT_TO_LANGUAGE = {
    ".py": Language.PYTHON,
    ".js": Language.JS,
    ".ts": Language.TS,
    ".cs": Language.CSHARP,
    ".sol": Language.SOL,
}


def split_source_file(path: Path, chunk_size: int = 1000, chunk_overlap: int = 100) -> list[Document]:
    language = EXT_TO_LANGUAGE[path.suffix]
    splitter = RecursiveCharacterTextSplitter.from_language(
        language=language, chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    doc = Document(
        page_content=path.read_text(encoding="utf-8"),
        metadata={"source": str(path), "language": language.value},
    )
    return splitter.split_documents([doc])


# 예시: 현재 폴더의 .py 파일을 분할 (있는 경우)
py_files = list(Path(".").glob("*.py"))
if py_files:
    print(split_source_file(py_files[0])[:2])
else:
    print("현재 폴더에 .py 파일이 없습니다.")